# Baseline model tests v2 (full datasets)

Notebook purpose:
- Train on full `labeled_msk_df.csv` and `labeled_regions_df.csv` by default.
- Use only `l1` and `l2` labels (no `l1_auto` / `l2_auto`).
- Use `train/val/test` split by `Source_File` to avoid leakage.
- Handle class imbalance with `class_weight='balanced'`.

Important `l2` note:
- Many objects legitimately have only `l1` and no `l2`.
- If we train a separate `l2` model, default behavior is to drop rows with empty `l2`.
- Optional mode `l2_missing_policy='as_class'` keeps them as `no_l2`.


In [ ]:
# If scikit-learn is missing, uncomment:
# !pip install -q scikit-learn


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, accuracy_score

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
rand_seed = 44


In [2]:
# Paths
base = Path(r'C:/Users/artem/MAIN/projects/plain/local/python_modules/scripts/geo')
msk_path = base / 'labeled_msk_df.csv'
regions_path = base / 'labeled_regions_df.csv'

# Full run by default
msk_samples_num = None
regions_samples_num = None

# Split settings
test_size = 0.2
val_size = 0.1  # fraction from train+val part

# Optional CV on train split only
apply_cross_val = False
cross_val_splits = 3

# l2 missing handling: 'drop' or 'as_class'
l2_missing_policy = 'as_class'


In [3]:
usecols = ['Source_File', 'Layer', 'Type', 'Text', 'l1', 'l2']

def load_df(path: Path, sample_n=None):
    print(f'Loading: {path}')
    df = pd.read_csv(path, usecols=usecols, low_memory=False)

    required = ['Source_File', 'Layer', 'Type', 'Text', 'l1', 'l2']
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f'Missing required columns: {missing}')

    for col in ['Source_File', 'Layer', 'Type', 'Text']:
        df[col] = df[col].fillna('').astype(str)

    # Keep l1/l2 nullable and handle missing explicitly in train_eval
    if sample_n is not None and len(df) > sample_n:
        df = df.sample(n=sample_n, random_state=rand_seed).reset_index(drop=True)

    label_l1 = 'l1'
    label_l2 = 'l2'

    print('Rows:', len(df))
    print('Unique Source_File:', df['Source_File'].nunique())
    print('Label columns:', label_l1, label_l2)
    return df, label_l1, label_l2

def report_missing(df: pd.DataFrame, name: str):
    l1_missing = df['l1'].isna().sum() + (df['l1'].astype(str).str.strip() == '').sum()
    l2_missing = df['l2'].isna().sum() + (df['l2'].astype(str).str.strip() == '').sum()
    print(f'\n{name} label coverage:')
    print(f'  l1 missing: {l1_missing:,} / {len(df):,} ({l1_missing / len(df) * 100:.2f}%)')
    print(f'  l2 missing: {l2_missing:,} / {len(df):,} ({l2_missing / len(df) * 100:.2f}%)')


In [4]:
def build_text_features(df: pd.DataFrame):
    return (
        'TYPE=' + df['Type'].astype(str) + ' ' +
        'LAYER=' + df['Layer'].astype(str) + ' ' +
        'TEXT=' + df['Text'].astype(str)
    )

def split_train_val_test(X, y, groups):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=rand_seed)
    train_val_idx, test_idx = next(gss.split(X, y, groups=groups))

    X_train_val = X.iloc[train_val_idx]
    y_train_val = y.iloc[train_val_idx]
    groups_train_val = groups.iloc[train_val_idx]

    X_test = X.iloc[test_idx]
    y_test = y.iloc[test_idx]
    groups_test = groups.iloc[test_idx]

    gss2 = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=rand_seed)
    train_idx, val_idx = next(gss2.split(X_train_val, y_train_val, groups=groups_train_val))

    X_train = X_train_val.iloc[train_idx]
    y_train = y_train_val.iloc[train_idx]
    groups_train = groups_train_val.iloc[train_idx]

    X_val = X_train_val.iloc[val_idx]
    y_val = y_train_val.iloc[val_idx]
    groups_val = groups_train_val.iloc[val_idx]

    return X_train, y_train, groups_train, X_val, y_val, groups_val, X_test, y_test, groups_test


In [5]:
def build_model():
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            analyzer='char_wb',
            ngram_range=(3, 5),
            min_df=2,
            max_features=200_000,
        )),
        ('clf', SGDClassifier(
            loss='log_loss',
            alpha=1e-5,
            max_iter=20,
            tol=1e-3,
            n_jobs=-1,
            random_state=rand_seed,
            class_weight='balanced'
        ))
    ])

def prepare_target(df: pd.DataFrame, label_col: str):
    if label_col == 'l1':
        y = df['l1']
        mask = y.notna() & (y.astype(str).str.strip() != '')
        dropped = (~mask).sum()
        return df.loc[mask].copy(), y.loc[mask].astype(str), dropped

    if label_col == 'l2':
        y = df['l2']
        if l2_missing_policy == 'drop':
            mask = y.notna() & (y.astype(str).str.strip() != '')
            dropped = (~mask).sum()
            return df.loc[mask].copy(), y.loc[mask].astype(str), dropped
        if l2_missing_policy == 'as_class':
            y2 = y.fillna('no_l2').astype(str).str.strip()
            y2 = y2.replace('', 'no_l2')
            dropped = 0
            return df.copy(), y2, dropped

    raise ValueError(f'Unsupported label_col: {label_col}')

def eval_split(y_true, y_pred, split_name):
    macro = f1_score(y_true, y_pred, average='macro')
    weighted = f1_score(y_true, y_pred, average='weighted')
    acc = accuracy_score(y_true, y_pred)
    print(f'{split_name:<6} | Macro-F1: {macro:.4f} | Weighted-F1: {weighted:.4f} | Acc: {acc:.4f}')

def train_eval(df: pd.DataFrame, label_col: str, name: str):
    data, y, dropped = prepare_target(df, label_col)

    X_text = build_text_features(data)
    groups = data['Source_File']

    X_train, y_train, groups_train, X_val, y_val, groups_val, X_test, y_test, groups_test = split_train_val_test(
        X_text, y, groups
    )

    model = build_model()
    model.fit(X_train, y_train)

    print(f'\n==== {name} | {label_col} ====')
    print(f'Total rows used: {len(data):,} | Dropped due to empty label: {dropped:,}')
    print(f'Classes: {y.nunique()}')
    print(f'Train size: {len(X_train):,} | Val size: {len(X_val):,} | Test size: {len(X_test):,}')

    eval_split(y_train, model.predict(X_train), 'train')
    eval_split(y_val, model.predict(X_val), 'val')
    eval_split(y_test, model.predict(X_test), 'test')

    if apply_cross_val:
        print('\nRunning GroupKFold CV on train split...')
        gkf = GroupKFold(n_splits=cross_val_splits)
        cv_macros = []
        for i, (tr_idx, va_idx) in enumerate(gkf.split(X_train, y_train, groups_train), 1):
            m = build_model()
            m.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
            preds = m.predict(X_train.iloc[va_idx])
            macro = f1_score(y_train.iloc[va_idx], preds, average='macro')
            cv_macros.append(macro)
            print(f'  Fold {i}: Macro-F1 = {macro:.4f}')
        print(f'CV Macro-F1 mean: {np.mean(cv_macros):.4f} | std: {np.std(cv_macros):.4f}')

    return model


In [6]:
# Moscow
msk_df, msk_l1, msk_l2 = load_df(msk_path, sample_n=msk_samples_num)
report_missing(msk_df, 'Moscow')

model_msk_l1 = train_eval(msk_df, msk_l1, 'Moscow')
model_msk_l2 = train_eval(msk_df, msk_l2, 'Moscow')


Loading: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\labeled_msk_df.csv
Rows: 180953
Unique Source_File: 370
Label columns: l1 l2

Moscow label coverage:
  l1 missing: 0 / 180,953 (0.00%)
  l2 missing: 22,551 / 180,953 (12.46%)

==== Moscow | l1 ====
Total rows used: 180,953 | Dropped due to empty label: 0
Classes: 12
Train size: 104,953 | Val size: 5,886 | Test size: 70,114
train  | Macro-F1: 0.9753 | Weighted-F1: 0.9977 | Acc: 0.9977
val    | Macro-F1: 0.9008 | Weighted-F1: 0.9959 | Acc: 0.9958
test   | Macro-F1: 0.9601 | Weighted-F1: 0.9916 | Acc: 0.9915

==== Moscow | l2 ====
Total rows used: 180,953 | Dropped due to empty label: 0
Classes: 32
Train size: 104,953 | Val size: 5,886 | Test size: 70,114
train  | Macro-F1: 0.9742 | Weighted-F1: 0.9962 | Acc: 0.9962
val    | Macro-F1: 0.9191 | Weighted-F1: 0.9961 | Acc: 0.9961
test   | Macro-F1: 0.9219 | Weighted-F1: 0.9885 | Acc: 0.9881


In [7]:
# Regions
reg_df, reg_l1, reg_l2 = load_df(regions_path, sample_n=regions_samples_num)
report_missing(reg_df, 'Regions')

model_reg_l1 = train_eval(reg_df, reg_l1, 'Regions')
model_reg_l2 = train_eval(reg_df, reg_l2, 'Regions')


Loading: C:\Users\artem\MAIN\projects\plain\local\python_modules\scripts\geo\labeled_regions_df.csv
Rows: 3069680
Unique Source_File: 146
Label columns: l1 l2

Regions label coverage:
  l1 missing: 0 / 3,069,680 (0.00%)
  l2 missing: 594,241 / 3,069,680 (19.36%)


c:\ProgramData\anaconda3\lib\site-packages\sklearn\linear_model\_stochastic_gradient.py:702: ConvergenceWarning: Maximum number of iteration reached before convergence. Consider increasing max_iter to improve the fit.
  warnings.warn(



==== Regions | l1 ====
Total rows used: 3,069,680 | Dropped due to empty label: 0
Classes: 24
Train size: 2,390,049 | Val size: 108,645 | Test size: 570,986
train  | Macro-F1: 0.4730 | Weighted-F1: 0.9442 | Acc: 0.9311
val    | Macro-F1: 0.4237 | Weighted-F1: 0.9026 | Acc: 0.8941
test   | Macro-F1: 0.3926 | Weighted-F1: 0.9110 | Acc: 0.8787

==== Regions | l2 ====
Total rows used: 3,069,680 | Dropped due to empty label: 0
Classes: 167
Train size: 2,390,049 | Val size: 108,645 | Test size: 570,986
train  | Macro-F1: 0.4682 | Weighted-F1: 0.9286 | Acc: 0.9247
val    | Macro-F1: 0.3960 | Weighted-F1: 0.8780 | Acc: 0.8748
test   | Macro-F1: 0.3315 | Weighted-F1: 0.8910 | Acc: 0.8895


## Interpretation guide
- Compare Macro-F1 first (important for minority classes).
- If train is high but val/test much lower, labels or domain shift are likely issues.
- For `l2`, lower metrics are expected because it is finer-grained and sparse.
